# Qwen single-agent TEC tool-calling experiment


## 1. Clean Colab Setup

This notebook is intended for Google Colab.

- Run the clean setup cell first.
- After it finishes, use Runtime -> Restart runtime.
- Then continue from the import check cell.
- Change only the CONFIG cell for normal experiments.
- This notebook does not rebuild IONEX data; it expects a processed parquet file.


In [ ]:
!pip uninstall -y transformers scikit-learn scipy
!pip install -U accelerate bitsandbytes torchvision pillow sentencepiece protobuf safetensors huggingface_hub pandas pyarrow pydantic python-dateutil
!pip install "transformers @ git+https://github.com/huggingface/transformers.git@main"


IMPORTANT:

After this cell finishes, restart the Colab runtime:

Runtime -> Restart runtime

Do not import transformers before restarting.


## 2. Import Check

Do not import scipy/sklearn in this notebook.


In [ ]:
import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
print("transformers Qwen imports OK")


## 3. CONFIG

For normal runs, change `SELECTED_PRESET` only. You can also edit the preset values here if you want a custom scenario.

The interval convention is `[START, END)`: `START` is included and `END` is excluded.


In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-4B"

LOAD_IN_4BIT = False
LOAD_IN_8BIT = False
TORCH_DTYPE = "float16"
MAX_INPUT_TOKENS = 4096

DATASET_REF = "default"
DATASET_FILENAME = "tec_regions_2024_01_01_to_2024_04_01_hourly.parquet"

START = "2024-03-01"
END = "2024-04-01"  # exclusive end

STATE_FEEDBACK_MODE = "state_aware"

MAX_STEPS = 12
MAX_TOOL_CALLS = 8

REGION_ID = "midlat_europe"
REGION_IDS = ("midlat_europe", "highlat_north")
Q = 0.9

TASK_PRESETS = {
    "hightec_midlat_europe": {
        "task_type": "high_tec",
        "region_id": "midlat_europe",
        "region_ids": ("midlat_europe",),
        "q": 0.9,
    },
    "hightec_highlat_north": {
        "task_type": "high_tec",
        "region_id": "highlat_north",
        "region_ids": ("highlat_north",),
        "q": 0.9,
    },
    "stable_midlat_europe": {
        "task_type": "stable_intervals",
        "region_id": "midlat_europe",
        "region_ids": ("midlat_europe",),
    },
    "compare_midlat_europe_highlat_north": {
        "task_type": "compare_regions",
        "region_id": None,
        "region_ids": ("midlat_europe", "highlat_north"),
    },
    "compare_three_regions": {
        "task_type": "compare_regions",
        "region_id": None,
        "region_ids": ("midlat_europe", "highlat_north", "equatorial_africa"),
    },
}

SELECTED_PRESET = "stable_midlat_europe"

BASE_CONFIG = {
    "model_name": MODEL_NAME,
    "load_in_4bit": LOAD_IN_4BIT,
    "load_in_8bit": LOAD_IN_8BIT,
    "torch_dtype": TORCH_DTYPE,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "dataset_ref": DATASET_REF,
    "dataset_filename": DATASET_FILENAME,
    "start": START,
    "end": END,
    "state_feedback_mode": STATE_FEEDBACK_MODE,
    "max_steps": MAX_STEPS,
    "max_tool_calls": MAX_TOOL_CALLS,
    "region_id": REGION_ID,
    "region_ids": REGION_IDS,
    "q": Q,
}

TASK_CONFIG = {**BASE_CONFIG, **TASK_PRESETS[SELECTED_PRESET]}
TASK_TYPE = TASK_CONFIG["task_type"]

print("selected_preset:", SELECTED_PRESET)
print("task_config:")
for key, value in TASK_CONFIG.items():
    print(f"  {key}: {value}")


## 4. Clone Or Update Repository


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/tec_agent_project")
REPO_URL = "https://github.com/ilyakosilov/tec_agent_project.git"

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "origin", "main"], check=True)

os.chdir(PROJECT_DIR)

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Working directory:", Path.cwd())
print("Latest commit:")
subprocess.run(["git", "log", "--oneline", "-3"], check=False)


## 5. Experiment Helpers

These helper functions are driven by `TASK_CONFIG`. They build the natural-language query, expected primitive tool sequence, EvalTask, metrics display, and final verdict.


In [ ]:
from calendar import month_name
from datetime import date

EPS = 1e-9


def _parse_date(value):
    return date.fromisoformat(value)


def _is_full_month(start, end):
    start_date = _parse_date(start)
    end_date = _parse_date(end)
    if start_date.day != 1:
        return False
    if start_date.month == 12:
        expected_end = date(start_date.year + 1, 1, 1)
    else:
        expected_end = date(start_date.year, start_date.month + 1, 1)
    return end_date == expected_end


def _period_phrase(config):
    start = config["start"]
    end = config["end"]
    if _is_full_month(start, end):
        start_date = _parse_date(start)
        return f"in {month_name[start_date.month]} {start_date.year}"
    return f"from {start} to {end} using exclusive end date"


def _join_regions(region_ids):
    region_ids = tuple(region_ids)
    if len(region_ids) == 1:
        return region_ids[0]
    if len(region_ids) == 2:
        return f"{region_ids[0]} and {region_ids[1]}"
    return ", ".join(region_ids[:-1]) + f", and {region_ids[-1]}"


def build_query(config):
    task_type = config["task_type"]
    period = _period_phrase(config)
    if task_type == "high_tec":
        return f"Find high TEC intervals for {config['region_id']} {period} with q={config.get('q', 0.9)}"
    if task_type == "stable_intervals":
        return f"Find stable TEC intervals for {config['region_id']} {period}"
    if task_type == "compare_regions":
        return f"Compare TEC statistics for {_join_regions(config['region_ids'])} {period}"
    if task_type == "report":
        return f"Build a TEC report for {_join_regions(config['region_ids'])} {period}"
    raise ValueError(f"Unsupported task_type: {task_type}")


def build_expected_tool_sequence(config):
    task_type = config["task_type"]
    if task_type == "high_tec":
        return [
            "tec_get_timeseries",
            "tec_compute_high_threshold",
            "tec_detect_high_intervals",
        ]
    if task_type == "stable_intervals":
        return [
            "tec_get_timeseries",
            "tec_compute_stability_thresholds",
            "tec_detect_stable_intervals",
        ]
    if task_type == "compare_regions":
        n_regions = len(config["region_ids"])
        return (
            ["tec_get_timeseries"] * n_regions
            + ["tec_compute_series_stats"] * n_regions
            + ["tec_compare_stats"]
        )
    raise ValueError(f"Unsupported task_type for expected sequence: {task_type}")


def build_eval_task(config, query, expected_sequence):
    task_type = config["task_type"]
    common = dict(
        task_id=f"qwen_smoke_{SELECTED_PRESET}",
        query=query,
        task_type=task_type,
        dataset_ref=config["dataset_ref"],
        start=config["start"],
        end=config["end"],
        q=config.get("q", 0.9),
        expected_tool_sequence=tuple(expected_sequence),
    )
    if task_type in {"high_tec", "stable_intervals"}:
        return EvalTask(
            **common,
            region_id=config["region_id"],
            region_ids=(config["region_id"],),
        )
    if task_type == "compare_regions":
        return EvalTask(
            **common,
            region_id=None,
            region_ids=tuple(config["region_ids"]),
            params={"metrics": ["mean", "median", "min", "max", "std", "p90", "p95"]},
        )
    raise ValueError(f"Unsupported task_type for EvalTask: {task_type}")


def print_key_metrics(task_type, metrics):
    common_keys = [
        "tool_sequence_match",
        "start_date_match",
        "end_date_match",
        "date_parse_match",
        "expected_n_points",
        "agent_timeseries_n_points",
        "gold_timeseries_n_points",
        "timeseries_n_points_match",
        "region_parse_match",
        "region_set_match",
        "expected_region_id",
        "expected_region_ids",
        "actual_region_id",
        "actual_region_ids",
        "actual_region_source",
    ]
    high_keys = [
        "threshold_abs_error",
        "threshold_exact_match",
        "interval_count_error",
        "interval_count_match",
        "global_peak_abs_error",
        "global_peak_match",
        "top_interval_start_match",
        "top_interval_end_match",
    ]
    stable_keys = [
        "stable_interval_count_error",
        "stable_interval_count_match",
        "top_stable_interval_start_match",
        "top_stable_interval_end_match",
        "top_stable_duration_abs_error",
        "top_stable_mean_abs_error",
        "top_stable_std_abs_error",
    ]
    compare_keys = [
        "region_set_match",
        "agent_region_count",
        "gold_region_count",
        "shared_region_count",
        "compare_stats_present",
        "stats_tool_call_count",
        "compare_stats_region_count",
        "pairwise_delta_count",
        "expected_pairwise_delta_count",
        "pairwise_delta_count_match",
        "pairwise_delta_region_pair_match",
        "shared_pairwise_delta_count",
        "mean_abs_error_avg",
        "max_abs_error_avg",
        "p90_abs_error_avg",
        "mean_abs_error_max",
        "max_abs_error_max",
        "p90_abs_error_max",
        "mean_delta_abs_error_avg",
        "max_delta_abs_error_avg",
        "p90_delta_abs_error_avg",
        "mean_delta_abs_error_max",
        "max_delta_abs_error_max",
        "p90_delta_abs_error_max",
    ]
    task_keys = {
        "high_tec": high_keys,
        "stable_intervals": stable_keys,
        "compare_regions": compare_keys,
    }.get(task_type, [])
    for key in common_keys + task_keys:
        print(f"{key}:", metrics.get(key))


def _metric_is_zero(metrics, key):
    value = metrics.get(key)
    return value is not None and abs(float(value)) <= EPS


def build_verdict_checks(task_type, result, gold_metric_result, metrics, expected_sequence):
    checks = {
        "agent_success": bool(result.success),
        "sequence_match": result.tool_sequence == expected_sequence,
        "gold_metric_success": bool(gold_metric_result.success),
        "tool_sequence_match": metrics.get("tool_sequence_match") is True,
        "start_date_match": metrics.get("start_date_match") is True,
        "end_date_match": metrics.get("end_date_match") is True,
        "date_parse_match": metrics.get("date_parse_match") is True,
        "timeseries_n_points_match": metrics.get("timeseries_n_points_match") is True,
        "no_parse_errors": result.parse_error_count == 0,
        "no_stall": not getattr(result, "stalled_loop_detected", False),
        "no_artifact_failure": not getattr(result, "artifact_usage_failure", False),
    }
    if task_type == "high_tec":
        checks.update({
            "threshold_exact_match": metrics.get("threshold_exact_match") is True,
            "interval_count_match": metrics.get("interval_count_match") is True,
            "global_peak_match": metrics.get("global_peak_match") is True,
        })
    elif task_type == "stable_intervals":
        checks.update({
            "stable_interval_count_match": metrics.get("stable_interval_count_match") is True,
        })
    elif task_type == "compare_regions":
        checks.update({
            "region_set_match": metrics.get("region_set_match") is True,
            "pairwise_delta_count_match": metrics.get("pairwise_delta_count_match") is True,
            "compare_stats_present": metrics.get("compare_stats_present") is True,
            "mean_abs_error_avg_zero": _metric_is_zero(metrics, "mean_abs_error_avg"),
            "mean_abs_error_max_zero": _metric_is_zero(metrics, "mean_abs_error_max"),
            "p90_abs_error_avg_zero": _metric_is_zero(metrics, "p90_abs_error_avg"),
        })
    return checks


## 6. Dataset Setup

Required processed dataset is selected by `DATASET_FILENAME` in CONFIG.

This notebook does not rebuild IONEX data. Use `notebooks/01_build_tec_dataset.ipynb` to build the processed parquet, then copy it here.


In [ ]:
from pathlib import Path
import shutil

DATASET_PATH = PROJECT_DIR / "data" / "processed" / TASK_CONFIG["dataset_filename"]
DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Expected dataset:", DATASET_PATH)
print("Exists:", DATASET_PATH.exists())
print("Dataset ref:", TASK_CONFIG["dataset_ref"])
print("Analysis period:", f"[{TASK_CONFIG['start']}, {TASK_CONFIG['end']})")


If the dataset is in Google Drive, uncomment and edit the next cell.


In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
#
# DRIVE_PARQUET_PATH = Path("/content/drive/MyDrive") / TASK_CONFIG["dataset_filename"]
# shutil.copy2(DRIVE_PARQUET_PATH, DATASET_PATH)
# print("Copied:", DATASET_PATH.exists(), DATASET_PATH)


## 7. Project Imports And Dataset Registration


In [ ]:
from tec_agents.agents.llm_single_agent import LLMSingleAgent
from tec_agents.data.datasets import get_dataset_summary, register_dataset
from tec_agents.eval.gold_runner import GoldRunner
from tec_agents.eval.metrics import compare_agent_to_gold
from tec_agents.eval.task_set import EvalTask, task_to_dict
from tec_agents.llm.local_qwen import LocalQwenChatModel
from tec_agents.mcp.client import LocalMCPClient
from tec_agents.mcp.server import build_local_mcp_server

assert DATASET_PATH.exists(), (
    f"Missing dataset: {DATASET_PATH}. "
    "Build it with notebooks/01_build_tec_dataset.ipynb or copy it from Drive."
)

register_dataset(
    dataset_ref=TASK_CONFIG["dataset_ref"],
    path=DATASET_PATH,
    file_format="parquet",
)

summary = get_dataset_summary(TASK_CONFIG["dataset_ref"])
summary


## 8. Initialize MCP-like Tools


In [ ]:
server = build_local_mcp_server(run_id=f"qwen_single_agent_{SELECTED_PRESET}_colab")
client = LocalMCPClient(server)

tool_names = client.list_tool_names()
print("Available tools:")
for name in tool_names:
    print("-", name)

required = {
    "tec_get_timeseries",
    "tec_compute_high_threshold",
    "tec_detect_high_intervals",
    "tec_compute_series_stats",
    "tec_compare_stats",
    "tec_compute_stability_thresholds",
    "tec_detect_stable_intervals",
}

print("Missing required tools:", sorted(required - set(tool_names)))
print("Legacy report tool present:", "tec_build_report" in tool_names)

assert "tec_build_report" not in tool_names
assert "tec_compare_regions" not in build_expected_tool_sequence(TASK_CONFIG)


## 9. Optional Hugging Face Login


In [ ]:
# Optional: use a Hugging Face token for higher rate limits or gated models.
# In Colab, add a secret named HF_TOKEN or HF_read_token.
try:
    from google.colab import userdata
    from huggingface_hub import login

    hf_token = userdata.get("HF_TOKEN") or userdata.get("HF_read_token")
    if hf_token:
        login(token=hf_token)
        print("Logged in to Hugging Face.")
    else:
        print("No HF token found. Continuing unauthenticated.")
except Exception as exc:
    print("HF login skipped:", exc)


## 10. Load Qwen

Default uses float16 without bitsandbytes 4-bit because some Colab runtimes fail with `libnvJitLink.so.13`. Change this in CONFIG if your runtime supports 4-bit.


In [ ]:
model = LocalQwenChatModel(
    model_name=TASK_CONFIG["model_name"],
    device_map="auto",
    torch_dtype=TASK_CONFIG["torch_dtype"],
    load_in_4bit=TASK_CONFIG["load_in_4bit"],
    load_in_8bit=TASK_CONFIG["load_in_8bit"],
    trust_remote_code=True,
    max_input_tokens=TASK_CONFIG["max_input_tokens"],
)


## 11. Run Agent

This uses state-aware free tool-calling: Qwen sees task state, artifacts, completed calls, and `[START, END)`, but still chooses each next tool itself.


In [ ]:
import json

query = build_query(TASK_CONFIG)
expected_sequence = build_expected_tool_sequence(TASK_CONFIG)

print("query:", query)
print("expected_sequence:", expected_sequence)
print("analysis_period:", f"[{TASK_CONFIG['start']}, {TASK_CONFIG['end']})")

agent = LLMSingleAgent(
    model=model,
    client=client,
    max_steps=TASK_CONFIG["max_steps"],
    max_tool_calls=TASK_CONFIG["max_tool_calls"],
    max_parse_retries=2,
    max_tool_retries=2,
    temperature=0.0,
    state_feedback_mode=TASK_CONFIG["state_feedback_mode"],
)

result = agent.run(query)
sequence_match = result.tool_sequence == expected_sequence

first_timeseries_call = next(
    (call for call in result.trace.get("calls", []) if call.get("tool_name") == "tec_get_timeseries"),
    None,
)
first_timeseries_args = (first_timeseries_call or {}).get("arguments") or {}
first_timeseries_output = (first_timeseries_call or {}).get("output") or {}
first_timeseries_metadata = first_timeseries_output.get("metadata") or {}

print("success:", result.success)
print("error_message:", result.error_message)
print("state_feedback_mode:", getattr(result, "state_feedback_mode", None))
print("tool_sequence:", result.tool_sequence)
print("sequence_match:", sequence_match)
print("parse_error_count:", result.parse_error_count)
print("invalid_json_count:", result.invalid_json_count)
print("unknown_format_count:", result.unknown_format_count)
print("repair_attempt_count:", result.repair_attempt_count)
print("repeated_tool_call_count:", getattr(result, "repeated_tool_call_count", None))
print("multi_tool_call_output_count:", getattr(result, "multi_tool_call_output_count", None))
print("multi_final_answer_output_count:", getattr(result, "multi_final_answer_output_count", None))
print("malformed_wrapper_count:", getattr(result, "malformed_wrapper_count", None))
print("stray_tool_call_tag_count:", getattr(result, "stray_tool_call_tag_count", None))
print("cleaned_output_changed_count:", getattr(result, "cleaned_output_changed_count", None))
print("stalled_loop_detected:", getattr(result, "stalled_loop_detected", None))
print("artifact_usage_failure:", getattr(result, "artifact_usage_failure", None))
print("date_range_mismatch_detected:", getattr(result, "date_range_mismatch_detected", None))
print("date_range_correction_count:", getattr(result, "date_range_correction_count", None))

print("expected_start:", TASK_CONFIG["start"])
print("expected_end:", TASK_CONFIG["end"])
print("actual first tool start:", first_timeseries_args.get("start"))
print("actual first tool end:", first_timeseries_args.get("end"))
print("n_points:", first_timeseries_metadata.get("n_points"))

print("completed_tool_calls:")
print(json.dumps(getattr(result, "completed_tool_calls", []), ensure_ascii=False, indent=2, default=str))
print("available_artifacts:")
print(json.dumps(getattr(result, "available_artifacts", {}), ensure_ascii=False, indent=2, default=str))
print("missing_goal_artifacts:", getattr(result, "missing_goal_artifacts", []))
print("state_snapshots:")
print(json.dumps(getattr(result, "state_snapshots", []), ensure_ascii=False, indent=2, default=str))

if getattr(result, "malformed_wrapper_count", 0):
    print("warning: model produced malformed wrapper tags, but cleaning may have recovered a valid block.")

print("answer:")
print(result.answer)


## 12. Compare With Deterministic Gold


In [ ]:
eval_task = build_eval_task(TASK_CONFIG, query, expected_sequence)

gold_result = GoldRunner().run(eval_task)
print("gold_status:", gold_result.status)
print("gold_error:", gold_result.error)
assert gold_result.status == "success", gold_result.error
assert gold_result.result is not None

gold_metric_result = compare_agent_to_gold(
    task_id=eval_task.task_id,
    task_type=eval_task.task_type,
    agent_result=result.tool_results,
    gold_result=gold_result.result,
    agent_trace=result.trace,
    task=task_to_dict(eval_task),
    parsed_task=result.parsed_task,
    orchestration_steps=result.orchestration_steps,
)

gold_comparison = gold_metric_result.metrics
print("gold_metric_success:", gold_metric_result.success)
print("gold_metric_errors:", gold_metric_result.errors)
print_key_metrics(TASK_TYPE, gold_comparison)


## 13. Final Verdict


In [ ]:
verdict_checks = build_verdict_checks(
    TASK_TYPE,
    result,
    gold_metric_result,
    gold_comparison,
    expected_sequence,
)
OVERALL_OK = all(verdict_checks.values())

print("verdict_checks:")
for key, value in verdict_checks.items():
    print(f"{key}: {value}")
print("OVERALL_OK:", OVERALL_OK)

wrapper_cleaned_ok = getattr(result, "malformed_wrapper_count", 0) == 0 or result.success
print("wrapper_cleaned_ok:", wrapper_cleaned_ok)
if getattr(result, "malformed_wrapper_count", 0):
    print("warning: wrapper anomalies were cleaned; inspect raw/cleaned outputs below.")


## 14. Inspect Raw And Cleaned Model Outputs


In [ ]:
for i, raw in enumerate(getattr(result, "raw_model_outputs", []), start=1):
    print("=" * 80)
    print("RAW MODEL OUTPUT", i)
    print(raw)

for i, cleaned in enumerate(getattr(result, "cleaned_model_outputs", []), start=1):
    print("=" * 80)
    print("CLEANED MODEL OUTPUT", i)
    print(cleaned)


## 15. Save Result JSON


In [ ]:
import json

output_path = PROJECT_DIR / "outputs" / "metrics" / f"qwen_single_agent_{SELECTED_PRESET}_colab.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

payload = {
    "selected_preset": SELECTED_PRESET,
    "task_config": TASK_CONFIG,
    "query": query,
    "expected_sequence": expected_sequence,
    "result": result.to_dict() if hasattr(result, "to_dict") else result.__dict__,
    "gold_comparison": gold_comparison,
    "verdict_checks": verdict_checks,
    "overall_ok": OVERALL_OK,
}

with output_path.open("w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2, default=str)

print("Saved:", output_path)


## 16. Optional Download Result


In [ ]:
# from google.colab import files
# files.download(str(output_path))
